# 模組一｜SAM 3 航照分割（Colab）

把國土測繪中心正射影像切成小圖，用 SAM 3 文字提示分割出鋪面、樹、植被、車道、行穿線，再推出人行道（鋪面－車道－行穿線）與灌木（植被－樹），輸出與原圖同座標的遮罩 GeoTIFF。

**執行前準備**
1. Runtime → Change runtime type → **T4 GPU**
2. 已在 Hugging Face 取得 [facebook/sam3](https://huggingface.co/facebook/sam3) 存取權限，並建立 Access Token（Read）
3. 把本機的 `data/processed/<id>_ortho_z19.tif` 上傳到 Google Drive 的 `taoyuan-corridor/` 資料夾

**輸出**：`taoyuan-corridor/<id>_mask_<類別>.tif`（下載回本機的 `data/processed/`，再執行 `corridor_metrics.py`）

In [ ]:
# ==== 參數（可調整）====
SITE_ID = "taoyuan_hospital"
DRIVE_DIR = "/content/drive/MyDrive/taoyuan-corridor"
# 類別：(提示詞, 分數門檻)。依 2026-11-16 桃園醫院中央小圖測試選定：
#   "sidewalk" 最高僅 0.37 分、"footpath" 幾乎抓不到 → 改用 "pavement"（鋪面，含車道）再扣掉 road／crosswalk
#   "trees" 比 "tree canopy" 分數高；"shrub" 抓不到 → 用 "vegetation" 扣掉 trees 當灌木草地
CLASSES = {
    "pavement":    ("pavement", 0.3),
    "tree_canopy": ("trees", 0.3),
    "vegetation":  ("vegetation", 0.3),
    "road":        ("road", 0.3),
    "crosswalk":   ("zebra crossing", 0.2),
}
SCORE_THRESHOLD = min(t for _, t in CLASSES.values())  # processor 先用最低門檻，各類別再各自過濾
CHIP = 1008             # 小圖邊長（px）
OVERLAP = 128           # 小圖重疊（px）


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 安裝 SAM 3 與地理套件。SAM 3 會把 numpy 降到 1.26，rasterio 固定 1.4.3（1.5 以上要求 numpy 2）
# 跑完後：執行階段 → 重新啟動工作階段，再從參數格往下跑（跳過本格）
!pip -q install "rasterio==1.4.3" "contourpy==1.3.3"
!git clone -q https://github.com/facebookresearch/sam3.git /content/sam3 || true
!pip -q install -e /content/sam3
import torch; print("torch", torch.__version__, "cuda", torch.cuda.is_available())

In [ ]:
# 登入 Hugging Face（貼上 Read 權限的 Access Token）
from huggingface_hub import login
login()

In [ ]:
import torch
import numpy as np, rasterio
from PIL import Image
from sam3.model_builder import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor

# 全程 float32：SAM 3 的 ViT MLP 第一層（perflib.fused.addmm_act）寫死轉 bfloat16，改成一般 float32 線性層
import torch.nn.functional as F
import sam3.model.vitdet as vitdet
def addmm_act_fp32(activation, linear, mat1):
    act = F.gelu if activation in (F.gelu, torch.nn.GELU) else F.relu
    return act(F.linear(mat1, linear.weight, linear.bias))
vitdet.addmm_act = addmm_act_fp32

# 明確指定 BPE 詞表路徑（Colab 的 /content 下有同名資料夾，pkg_resources 找不到套件位置）
model = build_sam3_image_model(bpe_path="/content/sam3/sam3/assets/bpe_simple_vocab_16e6.txt.gz")
model = model.float().eval()
# Sam3Processor 內建 confidence_threshold=0.5 會先濾掉低分實例，改成與 SCORE_THRESHOLD 一致
processor = Sam3Processor(model, confidence_threshold=SCORE_THRESHOLD)

src_path = f"{DRIVE_DIR}/{SITE_ID}_ortho_z19.tif"
with rasterio.open(src_path) as src:
    rgb = src.read().transpose(1, 2, 0)
    profile = src.profile
H, W = rgb.shape[:2]
print("影像大小", W, "x", H)

In [ ]:
def to_mask(m, shape):
    m = m.detach().float().cpu().numpy() if hasattr(m, "detach") else np.asarray(m, dtype=np.float32)
    m = np.squeeze(m)
    if m.shape != shape:  # 保險：尺寸不同時縮放回小圖大小
        m = np.asarray(Image.fromarray((m > 0.5).astype(np.uint8) * 255).resize(shape[::-1], Image.NEAREST)) / 255
    return m > 0.5

def chip_origins(n, chip, overlap):
    step = chip - overlap
    xs = list(range(0, max(n - chip, 0) + 1, step))
    if xs[-1] + chip < n:
        xs.append(n - chip)
    return xs

processor.set_confidence_threshold(SCORE_THRESHOLD)
masks = {k: np.zeros((H, W), dtype=bool) for k in CLASSES}
ys, xs = chip_origins(H, CHIP, OVERLAP), chip_origins(W, CHIP, OVERLAP)
total = len(ys) * len(xs)
done = 0
with torch.inference_mode():
    for y0 in ys:
        for x0 in xs:
            tile = rgb[y0:y0 + CHIP, x0:x0 + CHIP]
            state = processor.set_image(Image.fromarray(tile))
            for k, (prompt, thr) in CLASSES.items():
                out = processor.set_text_prompt(state=state, prompt=prompt)
                sc = out["scores"].detach().float().cpu().numpy()
                for m, s in zip(out["masks"], sc):
                    if float(s) >= thr:
                        masks[k][y0:y0 + CHIP, x0:x0 + CHIP] |= to_mask(m, tile.shape[:2])
            done += 1
            print(f"{done}/{total}", end="\r")

# 衍生類別（檔名對應本機 corridor_metrics.py / moi_sidewalk_metrics.py）
masks["sidewalk"] = masks["pavement"] & ~masks["road"] & ~masks["crosswalk"]
masks["shrub"] = masks["vegetation"] & ~masks["tree_canopy"]
for k, m in masks.items():
    print(f"{k:12s} 覆蓋 {m.mean():.1%}")
print(f"pavement 與 road 重疊比例 {(masks['pavement'] & masks['road']).sum() / max(masks['pavement'].sum(), 1):.0%}")


In [ ]:
# 寫出遮罩 GeoTIFF（與原圖同座標）
prof = profile.copy()
prof.update(count=1, dtype="uint8", compress="deflate", photometric="minisblack", nodata=None)
prof.pop("jpeg_quality", None)
for k, m in masks.items():
    out = f"{DRIVE_DIR}/{SITE_ID}_mask_{k}.tif"
    with rasterio.open(out, "w", **prof) as dst:
        dst.write(m.astype(np.uint8), 1)
    print("→", out)


In [ ]:
# 目視檢查：人行道（藍）、樹冠＋灌木（綠）、車道（紅）疊在航照上，取中央 1500px
import matplotlib.pyplot as plt
cy, cx, r = H // 2, W // 2, 750
win = (slice(cy - r, cy + r), slice(cx - r, cx + r))
view = rgb[win].astype(float)
for m, color, a in ((masks["road"][win], [255, 40, 40], 0.35),
                    ((masks["tree_canopy"] | masks["shrub"])[win], [0, 220, 60], 0.5),
                    (masks["sidewalk"][win], [30, 120, 255], 0.6)):
    view[m] = view[m] * (1 - a) + np.array(color) * a
plt.figure(figsize=(12, 12)); plt.imshow(view.clip(0, 255).astype(np.uint8)); plt.axis("off")
